# Vector DB benchmark + Simple RAG

**Part 1** (cell below): compare FAISS vs ChromaDB vs Qdrant — see `scripts/vector_db.py`.

**Part 2** (last cell): end-to-end RAG — whole-table chunks + Qdrant + Ollama via `src/rag/pipeline.py`.

We're comparing them on 3 simple things:

Build time — how long it takes to load all our chunks in
Search time — how long it takes to answer one question
Recall — does it still find the right answer (should be the same for all 3, since the math is the same — this is just a sanity check that we set each one up correctly)

In [2]:
# Simple test: load the same chunks into 3 different search tools,
# and compare how fast + accurate each one is.

import json
import time
import numpy as np
from sentence_transformers import SentenceTransformer

# force CPU — MPS (Apple Silicon GPU) crashes mid-encode with sentence-transformers
model = SentenceTransformer("BAAI/bge-small-en-v1.5", device="cpu")

# ---------- Step 1: load our winning chunks from Project 2 ----------
with open("/Users/saivardhannarla/projects/Mulit-Document Ingestions/data/processed/chunks_whole_table.jsonl") as f:
    chunks = [json.loads(l) for l in f]
chunks = [c for c in chunks if not c.get("is_noise", False)]

with open("/Users/saivardhannarla/projects/Mulit-Document Ingestions/data/processed/eval_dataset.jsonl") as f:
    eval_examples = [json.loads(l) for l in f]

with open("/Users/saivardhannarla/projects/Mulit-Document Ingestions/data/processed/chunks.jsonl") as f:
    row_level_chunks = [json.loads(l) for l in f]
gold_text_lookup = {c["chunk_id"]: c["text"] for c in row_level_chunks}

chunk_ids = [c["chunk_id"] for c in chunks]
chunk_texts = [c["text"] for c in chunks]

print(f"Loaded {len(chunks)} chunks and {len(eval_examples)} questions.")

# ---------- Step 2: turn all chunks and questions into numbers (embeddings) ----------
# we do this ONCE and reuse the same numbers for all 3 tools -- this keeps the test fair
print("Embedding chunks...")
chunk_vectors = model.encode(chunk_texts, normalize_embeddings=True, show_progress_bar=True)
chunk_vectors = np.array(chunk_vectors).astype("float32")

print("Embedding questions...")
question_texts = [ex["question"] for ex in eval_examples]
question_vectors = model.encode(question_texts, normalize_embeddings=True, show_progress_bar=True)
question_vectors = np.array(question_vectors).astype("float32")

dimension = chunk_vectors.shape[1]
print(f"Each embedding has {dimension} numbers.")


# ---------- Step 3: simple function to check if we found the right answer ----------
def contains_gold_info(retrieved_text, gold_text, threshold=0.6):
    gold_words = set(w.lower().strip(".,;:()") for w in gold_text.split())
    retrieved_words = set(w.lower().strip(".,;:()") for w in retrieved_text.split())
    if not gold_words:
        return False
    overlap = len(gold_words & retrieved_words) / len(gold_words)
    return overlap >= threshold


def check_recall(get_top5_texts_function):
    """get_top5_texts_function(question_index) -> list of 5 text strings"""
    found_count = 0
    for i, ex in enumerate(eval_examples):
        gold_ids = ex["gold_chunk_ids"]
        gold_texts = [gold_text_lookup[g] for g in gold_ids if g in gold_text_lookup]
        if not gold_texts:
            continue
        top5_texts = get_top5_texts_function(i)
        found = False
        for text in top5_texts:
            for gold_text in gold_texts:
                if contains_gold_info(text, gold_text):
                    found = True
        if found:
            found_count += 1
    return found_count / len(eval_examples)


results = {}

# ==================================================================
# TOOL 1: FAISS -- simplest, runs fully in memory, no setup needed
# ==================================================================
print("\n--- Testing FAISS ---")
import faiss

start_time = time.time()
faiss_index = faiss.IndexFlatIP(dimension)   # "IP" = inner product = cosine similarity (since our vectors are normalized)
faiss_index.add(chunk_vectors)
faiss_build_time = time.time() - start_time

start_time = time.time()
scores, indices = faiss_index.search(question_vectors, 5)
faiss_search_time = time.time() - start_time

def faiss_get_top5(question_index):
    top_indices = indices[question_index]
    return [chunk_texts[i] for i in top_indices]

faiss_recall = check_recall(faiss_get_top5)
results["FAISS"] = {"build_time": faiss_build_time, "search_time": faiss_search_time, "recall": faiss_recall}
print(f"build={faiss_build_time:.2f}s  search={faiss_search_time:.2f}s  recall={faiss_recall:.1%}")


# ==================================================================
# TOOL 2: ChromaDB -- local database, easy to use, saves to disk
# ==================================================================
print("\n--- Testing ChromaDB ---")
import chromadb

chroma_client = chromadb.PersistentClient(path="data/processed/chroma_db")
# delete old collection if it exists, so re-running this script doesn't crash
try:
    chroma_client.delete_collection("finqa_chunks")
except Exception:
    pass
chroma_collection = chroma_client.create_collection("finqa_chunks")

start_time = time.time()
# ChromaDB max batch size is 5461 — split into chunks of 5000
CHROMA_BATCH = 5000
for start in range(0, len(chunks), CHROMA_BATCH):
    end = start + CHROMA_BATCH
    chroma_collection.add(
        ids=chunk_ids[start:end],
        embeddings=chunk_vectors[start:end].tolist(),
        documents=chunk_texts[start:end],
    )
chroma_build_time = time.time() - start_time

start_time = time.time()
chroma_results_all = chroma_collection.query(
    query_embeddings=question_vectors.tolist(),
    n_results=5,
)
chroma_search_time = time.time() - start_time

def chroma_get_top5(question_index):
    return chroma_results_all["documents"][question_index]

chroma_recall = check_recall(chroma_get_top5)
results["ChromaDB"] = {"build_time": chroma_build_time, "search_time": chroma_search_time, "recall": chroma_recall}
print(f"build={chroma_build_time:.2f}s  search={chroma_search_time:.2f}s  recall={chroma_recall:.1%}")


# ==================================================================
# TOOL 3: Qdrant -- runs locally too (no server needed with path=)
# ==================================================================
print("\n--- Testing Qdrant ---")
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

qdrant_client = QdrantClient(path="data/processed/qdrant_db")
try:
    qdrant_client.delete_collection("finqa_chunks")
except Exception:
    pass
qdrant_client.create_collection(
    collection_name="finqa_chunks",
    vectors_config=VectorParams(size=dimension, distance=Distance.COSINE),
)

start_time = time.time()
points = []
for i in range(len(chunks)):
    points.append(PointStruct(
        id=i,
        vector=chunk_vectors[i].tolist(),
        payload={"text": chunk_texts[i]},
    ))
qdrant_client.upsert(collection_name="finqa_chunks", points=points)
qdrant_build_time = time.time() - start_time

start_time = time.time()
qdrant_all_results = []
for i in range(len(eval_examples)):
    hits = qdrant_client.query_points(
        collection_name="finqa_chunks",
        query=question_vectors[i].tolist(),
        limit=5,
    ).points
    qdrant_all_results.append(hits)
qdrant_search_time = time.time() - start_time

def qdrant_get_top5(question_index):
    hits = qdrant_all_results[question_index]
    return [hit.payload["text"] for hit in hits]

qdrant_recall = check_recall(qdrant_get_top5)
results["Qdrant"] = {"build_time": qdrant_build_time, "search_time": qdrant_search_time, "recall": qdrant_recall}
print(f"build={qdrant_build_time:.2f}s  search={qdrant_search_time:.2f}s  recall={qdrant_recall:.1%}")


# ==================================================================
# FINAL COMPARISON TABLE
# ==================================================================
print("\n--- FINAL COMPARISON ---")
print(f"{'Tool':<12}{'Build time':>12}{'Search time':>14}{'Recall':>10}")
for name, r in results.items():
    print(f"{name:<12}{r['build_time']:>11.2f}s{r['search_time']:>13.2f}s{r['recall']:>10.1%}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loaded 7176 chunks and 883 questions.
Embedding chunks...


Batches:   0%|          | 0/225 [00:00<?, ?it/s]

Embedding questions...


Batches:   0%|          | 0/28 [00:00<?, ?it/s]

Each embedding has 384 numbers.

--- Testing FAISS ---
build=0.01s  search=0.01s  recall=65.7%

--- Testing ChromaDB ---
build=1.24s  search=0.11s  recall=65.5%

--- Testing Qdrant ---
build=3.52s  search=3.48s  recall=65.7%

--- FINAL COMPARISON ---
Tool          Build time   Search time    Recall
FAISS              0.01s         0.01s     65.7%
ChromaDB           1.24s         0.11s     65.5%
Qdrant             3.52s         3.48s     65.7%


In [3]:
# Part 2: Simple RAG — question in, answer out
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.rag.pipeline import SimpleRAG

rag = SimpleRAG()
rag.setup()   # LangChain QdrantVectorStore + ChatOllama; run once to index

question = "what is the average payment volume per transaction for american express?"
result = rag.ask(question, top_k=5)

print("Question:", result["question"])
print("Answer:", result["answer"])
print("\nSources used:")
for i, src in enumerate(result["sources"], 1):
    print(f"  [{i}] {src[:100]}...")

rag.close()

Loading embedding model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding 7176 chunks...


Batches:   0%|          | 0/113 [00:00<?, ?it/s]

Building search index...
Setup done — index at /Users/saivardhannarla/projects/Mulit-Document Ingestions/data/processed/qdrant_db_final
Question: what is the average payment volume per transaction for american express?
Answer: 127.4

Sources used:
  [1] Company | Payments Volume (billions) | Total Volume (billions) | Total Transactions (billions) | Car...
  [2] under the terms of the american express settlement, mastercard is obligated to make 12 quarterly pay...
  [3] the quarterly payments will be in an amount equal to 15% (15% ) of american express 2019s u.s....
  [4] the amount of each quarterly payment is contingent on the performance of american express 2019s u.s....
  [5] mastercard 2019s maximum nominal payments will total $1800000....
